## NYISO leakage-safe load forecasts

This section selects the Hudson Valley load-forecast vintage that was available
when each NYISO day-ahead prediction would have been made.

For an electricity-delivery day \(D\), the prediction cutoff is defined as
5:00 a.m. Eastern Time on \(D-1\). Forecast vintages timestamped after this
cutoff are excluded to prevent the model from using information that would not
have been available at prediction time.

In [113]:
from pathlib import Path

import pandas as pd

# Starting from the current working folder, search upward through parent folders
# until the project root is found. The project root is identified by pyproject.toml.

def find_project_root(start: Path| None= None) -> Path:
    """Find the repository directory containing pyproject.toml."""
    if start is None:
        start = Path.cwd()

    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate

    # Stop with a clear error if the notebook is not running within this project.
    raise FileNotFoundError(
        "Could not find the project root containing pyproject.toml."
    )

# Resolve the project root so all later file paths are portable.
PROJECT_ROOT = find_project_root()

NYISO_ELECTRICITY_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "nyiso_hudson_valley_january_2025_electricity.csv"
)

# Define the cleaned NYISO electricity dataset produced by Notebook 02.
NYISO_FORECAST_VINTAGES_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "nyiso_hudson_valley_load_forecast_vintages.csv"
)

# Confirm both required inputs exist before feature engineering begins.
assert NYISO_ELECTRICITY_FILE.exists()
assert NYISO_FORECAST_VINTAGES_FILE.exists()

# Display the resolved file paths for reproducibility and troubleshooting.
print("Electricity:", NYISO_ELECTRICITY_FILE)
print("Forecasts:", NYISO_FORECAST_VINTAGES_FILE)

Electricity: C:\Users\david\Desktop\Data Science\DATA 698\electricity-price-forecasting\data\processed\nyiso_hudson_valley_january_2025_electricity.csv
Forecasts: C:\Users\david\Desktop\Data Science\DATA 698\electricity-price-forecasting\data\interim\nyiso_hudson_valley_load_forecast_vintages.csv


In [114]:
PJM_ELECTRICITY_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "pjm_pseg_january_2025_electricity.csv"
)

assert PJM_ELECTRICITY_FILE.exists()

print("PJM electricity:", PJM_ELECTRICITY_FILE)

PJM electricity: C:\Users\david\Desktop\Data Science\DATA 698\electricity-price-forecasting\data\processed\pjm_pseg_january_2025_electricity.csv


In [115]:
pjm_electricity = pd.read_csv(PJM_ELECTRICITY_FILE)

pjm_electricity["timestamp_utc"] = pd.to_datetime(
    pjm_electricity["timestamp_utc"],
    utc=True,
)

pjm_electricity["timestamp_local"] = (
    pd.to_datetime(
        pjm_electricity["timestamp_local"],
        utc=True,
    )
    .dt.tz_convert("America/New_York")
)

pjm_target_column = "day_ahead_price_usd_mwh"

assert pjm_target_column in pjm_electricity.columns

print("PJM electricity rows:", len(pjm_electricity))
print(
    "Unique PJM target hours:",
    pjm_electricity["timestamp_utc"].nunique(),
)
print("PJM target:", pjm_target_column)

assert len(pjm_electricity) == 744
assert pjm_electricity["timestamp_utc"].is_unique
assert pjm_electricity["timestamp_local"].is_unique
assert pjm_electricity["timestamp_utc"].is_monotonic_increasing
assert pjm_electricity[pjm_target_column].notna().all()

print("PJM input validation passed: 744 ordered, unique, nonmissing target hours.")

PJM electricity rows: 744
Unique PJM target hours: 744
PJM target: day_ahead_price_usd_mwh
PJM input validation passed: 744 ordered, unique, nonmissing target hours.


In [116]:
pjm_prediction_cutoff_hour = 11
provisional_price_schedule_availability_hour = 0

print(
    "PJM provisional prediction cutoff: D−1 "
    f"{pjm_prediction_cutoff_hour:02d}:00 America/New_York"
)
print(
    "Provisional price-schedule availability hour:",
    provisional_price_schedule_availability_hour,
)

PJM provisional prediction cutoff: D−1 11:00 America/New_York
Provisional price-schedule availability hour: 0


In [117]:
pjm_electricity["prediction_cutoff"] = (
    pjm_electricity["timestamp_local"].dt.normalize()
    - pd.DateOffset(days=1)
    + pd.Timedelta(hours=pjm_prediction_cutoff_hour)
)

pjm_electricity.loc[
    pjm_electricity["timestamp_local"]
    == pd.Timestamp(
        "2025-01-10 12:00",
        tz="America/New_York",
    ),
    [
        "timestamp_local",
        "prediction_cutoff",
    ],
]

,timestamp_local,prediction_cutoff
228,2025-01-10 12:00:00-05:00,2025-01-09 11:00:00-05:00


In [118]:
january_10_pjm_cutoffs = (
    pjm_electricity.loc[
        pjm_electricity["timestamp_local"].dt.date
        == pd.Timestamp("2025-01-10").date(),
        "prediction_cutoff",
    ]
    .drop_duplicates()
)

assert pjm_electricity["prediction_cutoff"].notna().all()
assert (
    pjm_electricity["prediction_cutoff"]
    < pjm_electricity["timestamp_local"]
).all()

assert january_10_pjm_cutoffs.tolist() == [
    pd.Timestamp(
        "2025-01-09 11:00",
        tz="America/New_York",
    )
]

print("PJM cutoff validation passed for all 744 target hours.")

PJM cutoff validation passed for all 744 target hours.


In [119]:
pjm_electricity["hour_of_day"] = (
    pjm_electricity["timestamp_local"].dt.hour
)

pjm_electricity["day_of_week"] = (
    pjm_electricity["timestamp_local"].dt.dayofweek
)

pjm_electricity["is_weekend"] = (
    pjm_electricity["day_of_week"].isin([5, 6])
)

assert pjm_electricity["hour_of_day"].between(0, 23).all()
assert pjm_electricity["day_of_week"].between(0, 6).all()
assert pjm_electricity["is_weekend"].notna().all()

print("PJM calendar features created.")

PJM calendar features created.


In [120]:
pjm_electricity["day_ahead_price_available_at"] = (
    pjm_electricity["timestamp_local"].dt.normalize()
    + pd.Timedelta(
        hours=provisional_price_schedule_availability_hour
    )
)

pjm_electricity[
    "target_day_price_available_by_cutoff"
] = (
    pjm_electricity["day_ahead_price_available_at"]
    <= pjm_electricity["prediction_cutoff"]
)

assert not pjm_electricity[
    "target_day_price_available_by_cutoff"
].any()



In [121]:
pjm_electricity["day_ahead_price_available_at"] = (
    pjm_electricity["timestamp_local"].dt.normalize()
    + pd.Timedelta(
        hours=provisional_price_schedule_availability_hour
    )
)

pjm_electricity[
    "target_day_price_available_by_cutoff"
] = (
    pjm_electricity["day_ahead_price_available_at"]
    <= pjm_electricity["prediction_cutoff"]
)

assert not pjm_electricity[
    "target_day_price_available_by_cutoff"
].any()

print(
    "Target-day PJM prices available by cutoff:",
    int(
        pjm_electricity[
            "target_day_price_available_by_cutoff"
        ].sum()
    ),
)

Target-day PJM prices available by cutoff: 0


In [122]:
pjm_electricity[
    "previous_day_same_hour_source_timestamp"
] = (
    pjm_electricity["timestamp_local"]
    - pd.DateOffset(days=1)
)

pjm_electricity.loc[
    pjm_electricity["timestamp_local"]
    == pd.Timestamp(
        "2025-01-10 12:00",
        tz="America/New_York",
    ),
    [
        "timestamp_local",
        "prediction_cutoff",
        "previous_day_same_hour_source_timestamp",
    ],
]

,timestamp_local,prediction_cutoff,previous_day_same_hour_source_timestamp
228,2025-01-10 12:00:00-05:00,2025-01-09 11:00:00-05:00,2025-01-09 12:00:00-05:00


In [123]:
#This is a SQL self-join: the table joins to itself where the source row’s timestamp equals 
# the target row’s prior-day same-hour 
# timestamp. validate="one_to_one" proves neither side duplicates an hourly key.

pjm_price_source_lookup = (
    pjm_electricity[
        [
            "timestamp_local",
            pjm_target_column,
            "day_ahead_price_available_at",
        ]
    ]
    .rename(
        columns={
            "timestamp_local": (
                "previous_day_same_hour_source_timestamp"
            ),
            pjm_target_column: (
                "previous_day_same_hour_price_value"
            ),
            "day_ahead_price_available_at": (
                "previous_day_same_hour_price_available_at"
            ),
        }
    )
)

pjm_with_price_sources = (
    pjm_electricity
    .merge(
        pjm_price_source_lookup,
        how="left",
        on="previous_day_same_hour_source_timestamp",
        validate="one_to_one",
    )
)

assert len(pjm_with_price_sources) == len(pjm_electricity)

print(
    "PJM price self-join rows:",
    len(pjm_with_price_sources),
)

PJM price self-join rows: 744


In [124]:
from electricity_forecasting.feature_engineering import (
    is_available_by_cutoff,
)

pjm_with_price_sources[
    "previous_day_same_hour_price_is_available"
] = is_available_by_cutoff(
    pjm_with_price_sources[
        "previous_day_same_hour_price_available_at"
    ],
    pjm_with_price_sources["prediction_cutoff"],
)

has_pjm_previous_day_source = pjm_with_price_sources[
    "previous_day_same_hour_price_value"
].notna()

assert pjm_with_price_sources.loc[
    has_pjm_previous_day_source,
    "previous_day_same_hour_price_is_available",
].all()

print(
    "PJM rows with a prior-day same-hour source:",
    int(has_pjm_previous_day_source.sum()),
)
print(
    "PJM availability-safe prior-day sources:",
    int(
        pjm_with_price_sources[
            "previous_day_same_hour_price_is_available"
        ].sum()
    ),
)

PJM rows with a prior-day same-hour source: 720
PJM availability-safe prior-day sources: 720


In [125]:
from electricity_forecasting.feature_engineering import (
    add_cutoff_safe_feature,
)

pjm_with_price_sources = add_cutoff_safe_feature(
    pjm_with_price_sources,
    source_value_column="previous_day_same_hour_price_value",
    source_available_at_column=(
        "previous_day_same_hour_price_available_at"
    ),
    prediction_cutoff_column="prediction_cutoff",
    feature_column="day_ahead_price_lag_1d",
)

assert (
    pjm_with_price_sources["day_ahead_price_lag_1d"].notna()
    == pjm_with_price_sources[
        "previous_day_same_hour_price_is_available"
    ]
).all()

print(
    "Nonmissing cutoff-safe PJM price lags:",
    int(
        pjm_with_price_sources[
            "day_ahead_price_lag_1d"
        ].notna().sum()
    ),
)

Nonmissing cutoff-safe PJM price lags: 720


In [126]:
from electricity_forecasting.feature_engineering import (
    add_rolling_mean_from_safe_feature,
)

pjm_with_price_sources = (
    add_rolling_mean_from_safe_feature(
        pjm_with_price_sources,
        safe_feature_column="day_ahead_price_lag_1d",
        window=24,
        feature_column=(
            "day_ahead_price_lag_1d_rolling_mean_24h"
        ),
    )
)

print(
    "Nonmissing 24-hour PJM rolling means:",
    int(
        pjm_with_price_sources[
            "day_ahead_price_lag_1d_rolling_mean_24h"
        ].notna().sum()
    ),
)

Nonmissing 24-hour PJM rolling means: 697


In [127]:
common_candidate_feature_columns = [
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "day_ahead_price_lag_1d",
    "day_ahead_price_lag_1d_rolling_mean_24h",
]

pjm_candidate_feature_columns = (
    common_candidate_feature_columns.copy()
)

nyiso_augmented_candidate_feature_columns = [
    *common_candidate_feature_columns,
    "load_forecast_mw",
]

assert set(pjm_candidate_feature_columns).issubset(
    pjm_with_price_sources.columns
)
assert "load_forecast_mw" not in pjm_candidate_feature_columns
assert set(nyiso_augmented_candidate_feature_columns) == (
    set(pjm_candidate_feature_columns)
    | {"load_forecast_mw"}
)

print("Common candidate features:", common_candidate_feature_columns)
print(
    "NYISO-only augmented feature:",
    "load_forecast_mw",
)

Common candidate features: ['hour_of_day', 'day_of_week', 'is_weekend', 'day_ahead_price_lag_1d', 'day_ahead_price_lag_1d_rolling_mean_24h']
NYISO-only augmented feature: load_forecast_mw


In [128]:
# PJM’s audit, identifier, and excluded-operational groups in a new cell.

pjm_price_feature_audit_columns = [
    "prediction_cutoff",
    "day_ahead_price_available_at",
    "target_day_price_available_by_cutoff",
    "previous_day_same_hour_source_timestamp",
    "previous_day_same_hour_price_value",
    "previous_day_same_hour_price_available_at",
    "previous_day_same_hour_price_is_available",
    "is_verified",
]

pjm_identifier_columns = [
    "timestamp_local",
    "timestamp_utc",
    "location_id",
    "location",
    "zone",
    "load_area",
    "weather_station",
    "market",
    "pricing_location",
    "load_data_type",
]

pjm_excluded_operational_columns = [
    "loss_component_usd_mwh",
    "congestion_component_usd_mwh",
    "energy_component_usd_mwh",
    "actual_load_mw",
    "observed_at",
    "REPORT_TYPE",
    "temperature_c",
    "dew_point_c",
    "relative_humidity_pct",
    "wind_speed_mps",
    "weather_quality_flagged",
    "weather_value_rejected",
    "weather_missing",
    "weather_imputed",
]

print("PJM audit columns:", pjm_price_feature_audit_columns)
print(
    "PJM excluded operational columns:",
    pjm_excluded_operational_columns,
)

PJM audit columns: ['prediction_cutoff', 'day_ahead_price_available_at', 'target_day_price_available_by_cutoff', 'previous_day_same_hour_source_timestamp', 'previous_day_same_hour_price_value', 'previous_day_same_hour_price_available_at', 'previous_day_same_hour_price_is_available', 'is_verified']
PJM excluded operational columns: ['loss_component_usd_mwh', 'congestion_component_usd_mwh', 'energy_component_usd_mwh', 'actual_load_mw', 'observed_at', 'REPORT_TYPE', 'temperature_c', 'dew_point_c', 'relative_humidity_pct', 'wind_speed_mps', 'weather_quality_flagged', 'weather_value_rejected', 'weather_missing', 'weather_imputed']


In [129]:
# alidate that every PJM group exists, has no duplicates, and does not overlap another group.

from itertools import combinations

pjm_column_groups = {
    "target": [pjm_target_column],
    "candidate_predictors": pjm_candidate_feature_columns,
    "audit": pjm_price_feature_audit_columns,
    "identifiers": pjm_identifier_columns,
    "excluded_operational": pjm_excluded_operational_columns,
}

pjm_dataset_columns = set(pjm_with_price_sources.columns)

for group_name, columns in pjm_column_groups.items():
    missing_columns = set(columns) - pjm_dataset_columns

    assert not missing_columns, (
        f"{group_name} contains missing columns: "
        f"{sorted(missing_columns)}"
    )
    assert len(columns) == len(set(columns)), (
        f"{group_name} contains duplicate column names"
    )

for left_group, right_group in combinations(
    pjm_column_groups,
    2,
):
    overlapping_columns = (
        set(pjm_column_groups[left_group])
        & set(pjm_column_groups[right_group])
    )

    assert not overlapping_columns, (
        f"{left_group} and {right_group} overlap: "
        f"{sorted(overlapping_columns)}"
    )

pjm_prohibited_predictors = (
    set(pjm_price_feature_audit_columns)
    | set(pjm_identifier_columns)
    | set(pjm_excluded_operational_columns)
    | {pjm_target_column}
)

assert set(pjm_candidate_feature_columns).isdisjoint(
    pjm_prohibited_predictors
)

print("PJM column-group validation passed.")

PJM column-group validation passed.


In [130]:
expected_pjm_source_timestamp = (
    pjm_with_price_sources["timestamp_local"]
    - pd.DateOffset(days=1)
)

assert pjm_with_price_sources[
    "previous_day_same_hour_source_timestamp"
].eq(expected_pjm_source_timestamp).all()

first_pjm_delivery_day = (
    pjm_with_price_sources["timestamp_local"].dt.normalize()
    == pd.Timestamp(
        "2025-01-01",
        tz="America/New_York",
    )
)

assert first_pjm_delivery_day.sum() == 24
assert pjm_with_price_sources.loc[
    first_pjm_delivery_day,
    "day_ahead_price_lag_1d",
].isna().all()
assert pjm_with_price_sources.loc[
    ~first_pjm_delivery_day,
    "day_ahead_price_lag_1d",
].notna().all()

assert (
    pjm_with_price_sources[
        "day_ahead_price_lag_1d_rolling_mean_24h"
    ].notna().sum()
    == 697
)

print(
    "PJM prior-day source timestamps verified for 744 target hours."
)
print(
    "PJM first delivery-day rows without a prior-day source:",
    int(first_pjm_delivery_day.sum()),
)
print("PJM full-window rolling means: 697")

PJM prior-day source timestamps verified for 744 target hours.
PJM first delivery-day rows without a prior-day source: 24
PJM full-window rolling means: 697


### Load the electricity and forecast-vintage data

The processed NYISO electricity data contain the hourly day-ahead LBMP target.
The interim forecast table contains all six Hudson Valley load-forecast vintages
for every January 2025 target hour.

Timestamp columns are parsed explicitly so availability comparisons are made
using timezone-aware values.

In [131]:
# Load the cleaned NYISO electricity data and all archived load-forecast
# vintages. Parse timestamps explicitly as timezone-aware values so forecast
# availability can be compared safely with each prediction cutoff.

nyiso_electricity = pd.read_csv(
    NYISO_ELECTRICITY_FILE
)

# Use the documented canonical target name. The conditional rename keeps
# this notebook compatible with an older processed-file alias.
if "day_ahead_lmp" in nyiso_electricity.columns:
    nyiso_electricity = nyiso_electricity.rename(
        columns={"day_ahead_lmp": "day_ahead_price_usd_mwh"}
    )

assert "day_ahead_price_usd_mwh" in nyiso_electricity.columns

forecast_vintages = pd.read_csv(
    NYISO_FORECAST_VINTAGES_FILE
)

nyiso_electricity["timestamp_utc"] = pd.to_datetime(
    nyiso_electricity["timestamp_utc"],
    utc=True,
)

nyiso_electricity["timestamp_local"] = (
    pd.to_datetime(
        nyiso_electricity["timestamp_local"],
        utc=True,
    )
    .dt.tz_convert("America/New_York")
)

forecast_vintages["target_timestamp"] = (
    pd.to_datetime(
        forecast_vintages["target_timestamp"],
        utc=True,
    )
    .dt.tz_convert("America/New_York")
)

forecast_vintages["forecast_available_at"] = (
    pd.to_datetime(
        forecast_vintages["forecast_available_at"],
        utc=True,
    )
    .dt.tz_convert("America/New_York")
)

print("Electricity rows:", len(nyiso_electricity))
print("Forecast-vintage rows:", len(forecast_vintages))
print(
    "Unique forecast target hours:",
    forecast_vintages["target_timestamp"].nunique(),
)

Electricity rows: 744
Forecast-vintage rows: 4464
Unique forecast target hours: 744


### Calculate the day-ahead prediction cutoff

For every target operating day, the prediction cutoff is 5:00 a.m. Eastern Time
on the preceding calendar day. All 24 target hours belonging to the same
operating day therefore share the same prediction cutoff.

The cutoff is constructed from local calendar dates so the logic remains valid
when the study is later expanded across daylight-saving-time transitions.

In [132]:
# Calculate one day-ahead prediction cutoff per target operating day:
# 5:00 a.m. Eastern Time on the preceding calendar day. All 24 delivery
# hours on the same operating date share this cutoff.

target_operating_dates = pd.to_datetime(
    forecast_vintages["target_timestamp"].dt.date
)

cutoff_dates = (
    target_operating_dates
    - pd.DateOffset(days=1)
    + pd.Timedelta(hours=5)
)

forecast_vintages["prediction_cutoff"] = (
    cutoff_dates.dt.tz_localize(
        "America/New_York",
        ambiguous="raise",
        nonexistent="raise",
    )
)

forecast_vintages[
    [
        "target_timestamp",
        "forecast_available_at",
        "prediction_cutoff",
        "source_file",
    ]
].head()

,target_timestamp,forecast_available_at,prediction_cutoff,source_file
0,2025-01-01 00:00:00-05:00,2024-12-26 07:05:02-05:00,2024-12-31 05:00:00-05:00,20241227isolf.csv
1,2025-01-01 00:00:00-05:00,2024-12-27 07:50:04-05:00,2024-12-31 05:00:00-05:00,20241228isolf.csv
2,2025-01-01 00:00:00-05:00,2024-12-28 07:50:08-05:00,2024-12-31 05:00:00-05:00,20241229isolf.csv
3,2025-01-01 00:00:00-05:00,2024-12-29 07:30:06-05:00,2024-12-31 05:00:00-05:00,20241230isolf.csv
4,2025-01-01 00:00:00-05:00,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,20241231isolf.csv


In [133]:
# Inspect the calculated cutoff for January 10 to confirm that every target
# hour on that operating day uses January 9 at 5:00 a.m. Eastern.

january_10_cutoffs = (
    forecast_vintages.loc[
        forecast_vintages["target_timestamp"].dt.date
        == pd.Timestamp("2025-01-10").date(),
        "prediction_cutoff",
    ]
    .drop_duplicates()
)

january_10_cutoffs

1296   2025-01-09 05:00:00-05:00
Name: prediction_cutoff, dtype: datetime64[us, America/New_York]

In [134]:
# Validate the cutoff calculation: retain all forecast-vintage rows, cover
# all 744 January target hours, and confirm the expected January 10 cutoff.

assert len(forecast_vintages) == 4464
assert forecast_vintages["target_timestamp"].nunique() == 744
assert forecast_vintages["prediction_cutoff"].notna().all()

assert january_10_cutoffs.tolist() == [
    pd.Timestamp(
        "2025-01-09 05:00",
        tz="America/New_York",
    )
]

print("Step 4 passed: prediction cutoffs calculated correctly.")

Step 4 passed: prediction cutoffs calculated correctly.


### Identify forecast vintages available by the cutoff

A forecast vintage is eligible only when its recorded availability timestamp is
at or before the 5:00 a.m. Eastern Time prediction cutoff for the target
operating day.

Forecasts published after the cutoff are excluded because they would not have
been available when the day-ahead prediction was made.

In [135]:
# Keep only forecast vintages that were available at or before each target
# hour's prediction cutoff. This prevents later forecast revisions from
# leaking future information into the day-ahead model.

eligible_vintages = forecast_vintages.loc[
    forecast_vintages["forecast_available_at"]
    <= forecast_vintages["prediction_cutoff"]
].copy()

eligible_counts = (
    eligible_vintages
    .groupby("target_timestamp")
    .size()
)

assert eligible_counts.size == 744
assert eligible_counts.ge(1).all()

assert (
    eligible_vintages["forecast_available_at"]
    <= eligible_vintages["prediction_cutoff"]
).all()

print("Eligible forecast rows:", len(eligible_vintages))
print(
    "Eligible vintages per target hour:",
    eligible_counts.value_counts().sort_index().to_dict(),
)

Eligible forecast rows: 3720
Eligible vintages per target hour: {5: 744}


### Select the latest eligible forecast

For each target operating hour, the eligible vintage with the most recent
availability timestamp is selected.

A tie at the latest availability timestamp is treated as a data-quality error,
rather than being silently resolved. The selected rows retain their forecast,
source, and availability provenance for auditability.

In [136]:
# For each target hour, select the most recent forecast that was still
# available by the cutoff. Preserve source and availability fields so the
# selected forecast can be audited later.

eligible_vintages["latest_eligible_available_at"] = (
    eligible_vintages
    .groupby("target_timestamp")["forecast_available_at"]
    .transform("max")
)

latest_eligible_rows = eligible_vintages.loc[
    eligible_vintages["forecast_available_at"]
    == eligible_vintages["latest_eligible_available_at"]
].copy()

latest_row_counts = (
    latest_eligible_rows
    .groupby("target_timestamp")
    .size()
)

assert latest_row_counts.size == 744
assert latest_row_counts.eq(1).all(), (
    "Expected exactly one latest eligible vintage per target hour; "
    f"found ties for {latest_row_counts[latest_row_counts.ne(1)].index.tolist()}"
)

selected_forecast_columns = [
    "target_timestamp",
    "load_forecast_mw",
    "forecast_available_at",
    "prediction_cutoff",
    "source_archive",
    "source_file",
    "availability_basis",
    "availability_is_proxy",
]

if "forecast_horizon_hours" in latest_eligible_rows.columns:
    selected_forecast_columns.append("forecast_horizon_hours")

selected_load_forecasts = (
    latest_eligible_rows
    .loc[:, selected_forecast_columns]
    .sort_values("target_timestamp")
    .reset_index(drop=True)
)

selected_load_forecasts["hours_before_cutoff"] = (
    selected_load_forecasts["prediction_cutoff"]
    - selected_load_forecasts["forecast_available_at"]
).dt.total_seconds() / 3600

selected_load_forecasts[
    [
        "target_timestamp",
        "load_forecast_mw",
        "forecast_available_at",
        "prediction_cutoff",
        "hours_before_cutoff",
        "source_archive",
        "source_file",
    ]
].head()

,target_timestamp,load_forecast_mw,forecast_available_at,prediction_cutoff,hours_before_cutoff,source_archive,source_file
0,2025-01-01 00:00:00-05:00,930,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,21.332222,20241201isolf_csv.zip,20241231isolf.csv
1,2025-01-01 01:00:00-05:00,889,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,21.332222,20241201isolf_csv.zip,20241231isolf.csv
2,2025-01-01 02:00:00-05:00,855,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,21.332222,20241201isolf_csv.zip,20241231isolf.csv
3,2025-01-01 03:00:00-05:00,841,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,21.332222,20241201isolf_csv.zip,20241231isolf.csv
4,2025-01-01 04:00:00-05:00,839,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,21.332222,20241201isolf_csv.zip,20241231isolf.csv


### Verify the selected forecast vintages

The selected dataset should contain one forecast for each of the 744 January
target hours. Every selected forecast must have been available by its prediction
cutoff.

January 10 is inspected separately because its latest source report,
`20250110isolf.csv`, became available after the January 9 cutoff. The expected
leakage-safe source is therefore `20250109isolf.csv`.

In [137]:
# Validate that exactly one eligible forecast was selected for every January
# target hour and that no selected forecast was available after its cutoff.
# Confirm January 10 correctly uses the January 9 source report.

january_10_selected = selected_load_forecasts.loc[
    selected_load_forecasts["target_timestamp"].dt.date
    == pd.Timestamp("2025-01-10").date(),
    [
        "target_timestamp",
        "load_forecast_mw",
        "forecast_available_at",
        "prediction_cutoff",
        "hours_before_cutoff",
        "source_file",
    ],
]

january_10_selected.head()

,target_timestamp,load_forecast_mw,forecast_available_at,prediction_cutoff,hours_before_cutoff,source_file
216,2025-01-10 00:00:00-05:00,1117,2025-01-08 07:35:04-05:00,2025-01-09 05:00:00-05:00,21.415556,20250109isolf.csv
217,2025-01-10 01:00:00-05:00,1091,2025-01-08 07:35:04-05:00,2025-01-09 05:00:00-05:00,21.415556,20250109isolf.csv
218,2025-01-10 02:00:00-05:00,1077,2025-01-08 07:35:04-05:00,2025-01-09 05:00:00-05:00,21.415556,20250109isolf.csv
219,2025-01-10 03:00:00-05:00,1074,2025-01-08 07:35:04-05:00,2025-01-09 05:00:00-05:00,21.415556,20250109isolf.csv
220,2025-01-10 04:00:00-05:00,1086,2025-01-08 07:35:04-05:00,2025-01-09 05:00:00-05:00,21.415556,20250109isolf.csv


In [138]:
# Confirm electricity hours and selected forecast target hours align exactly,
# then merge one leakage-safe load forecast into each NYISO electricity row.
# A one-to-one merge prevents duplicated or unmatched delivery hours.

assert len(selected_load_forecasts) == 744

assert selected_load_forecasts["target_timestamp"].nunique() == 744

assert not selected_load_forecasts.duplicated(
    "target_timestamp"
).any()

assert selected_load_forecasts["load_forecast_mw"].notna().all()

assert (
    selected_load_forecasts["forecast_available_at"]
    <= selected_load_forecasts["prediction_cutoff"]
).all()

assert selected_load_forecasts["hours_before_cutoff"].ge(0).all()

expected_latest_times = (
    eligible_vintages
    .groupby("target_timestamp")["forecast_available_at"]
    .max()
    .sort_index()
)

actual_latest_times = (
    selected_load_forecasts
    .set_index("target_timestamp")["forecast_available_at"]
    .sort_index()
)

pd.testing.assert_series_equal(
    actual_latest_times,
    expected_latest_times,
    check_names=False,
)

assert len(january_10_selected) == 24

assert set(january_10_selected["source_file"]) == {
    "20250109isolf.csv"
}

print("Step 5 passed: one latest eligible forecast selected for each target hour.")

Step 5 passed: one latest eligible forecast selected for each target hour.


## Step 6: Merge the leakage-safe load forecasts

Both `target_timestamp` and `timestamp_local` represent the beginning of the
same America/New_York delivery hour. Their exact one-to-one correspondence is
validated before the merge.

The left merge preserves all 744 NYISO electricity target hours and carries the
selected forecast's cutoff and source-provenance fields forward for auditability.

In [139]:
# Validate the merged modeling dataframe: 744 unique hourly rows, matching
# delivery timestamps, nonmissing selected forecasts, and complete forecast
# provenance showing every forecast was available by its prediction cutoff.

electricity_target_hours = pd.DatetimeIndex(
    nyiso_electricity["timestamp_local"]
).sort_values()
forecast_target_hours = pd.DatetimeIndex(
    selected_load_forecasts["target_timestamp"]
).sort_values()

assert electricity_target_hours.tz is not None
assert forecast_target_hours.tz is not None

pd.testing.assert_index_equal(
    electricity_target_hours,
    forecast_target_hours,
    check_names=False,
)

nyiso_electricity_with_load_forecasts = (
    nyiso_electricity
    .merge(
        selected_load_forecasts,
        how="left",
        left_on="timestamp_local",
        right_on="target_timestamp",
        validate="one_to_one",
    )
    .sort_values("timestamp_utc")
    .reset_index(drop=True)
)

nyiso_electricity_with_load_forecasts.head()

,timestamp_local,timestamp_utc,location_id,location,day_ahead_price_usd_mwh,loss_component_usd_mwh,congestion_component_usd_mwh,energy_component_usd_mwh,source_time_zone,load_location,...,target_timestamp,load_forecast_mw,forecast_available_at,prediction_cutoff,source_archive,source_file,availability_basis,availability_is_proxy,forecast_horizon_hours,hours_before_cutoff
0,2025-01-01 00:00:00-05:00,2025-01-01 05:00:00+00:00,61758,HUD VL,33.16,1.31,0.0,31.85,EST,HUD VL,...,2025-01-01 00:00:00-05:00,930,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,20241201isolf_csv.zip,20241231isolf.csv,zip_entry_last_modified,True,40.332222,21.332222
1,2025-01-01 01:00:00-05:00,2025-01-01 06:00:00+00:00,61758,HUD VL,32.07,1.26,0.0,30.81,EST,HUD VL,...,2025-01-01 01:00:00-05:00,889,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,20241201isolf_csv.zip,20241231isolf.csv,zip_entry_last_modified,True,41.332222,21.332222
2,2025-01-01 02:00:00-05:00,2025-01-01 07:00:00+00:00,61758,HUD VL,30.02,1.16,0.0,28.86,EST,HUD VL,...,2025-01-01 02:00:00-05:00,855,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,20241201isolf_csv.zip,20241231isolf.csv,zip_entry_last_modified,True,42.332222,21.332222
3,2025-01-01 03:00:00-05:00,2025-01-01 08:00:00+00:00,61758,HUD VL,28.28,1.01,0.0,27.27,EST,HUD VL,...,2025-01-01 03:00:00-05:00,841,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,20241201isolf_csv.zip,20241231isolf.csv,zip_entry_last_modified,True,43.332222,21.332222
4,2025-01-01 04:00:00-05:00,2025-01-01 09:00:00+00:00,61758,HUD VL,28.22,1.03,0.0,27.19,EST,HUD VL,...,2025-01-01 04:00:00-05:00,839,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,20241201isolf_csv.zip,20241231isolf.csv,zip_entry_last_modified,True,44.332222,21.332222


In [140]:
assert len(nyiso_electricity_with_load_forecasts) == 744
assert nyiso_electricity_with_load_forecasts["timestamp_local"].is_unique
assert nyiso_electricity_with_load_forecasts["target_timestamp"].is_unique

pd.testing.assert_series_equal(
    nyiso_electricity_with_load_forecasts["timestamp_local"],
    nyiso_electricity_with_load_forecasts["target_timestamp"],
    check_names=False,
)

assert nyiso_electricity_with_load_forecasts["load_forecast_mw"].notna().all()

forecast_audit_columns = [
    "forecast_available_at",
    "prediction_cutoff",
    "hours_before_cutoff",
    "source_archive",
    "source_file",
    "availability_basis",
    "availability_is_proxy",
]

assert set(forecast_audit_columns).issubset(
    nyiso_electricity_with_load_forecasts.columns
)

assert (
    nyiso_electricity_with_load_forecasts["forecast_available_at"]
    <= nyiso_electricity_with_load_forecasts["prediction_cutoff"]
).all()

print("Step 6 passed: leakage-safe load forecasts merged one-to-one with NYISO electricity.")

Step 6 passed: leakage-safe load forecasts merged one-to-one with NYISO electricity.


### Classify NYISO modeling columns

The model-column classification is:

- **Prediction target:** `day_ahead_price_usd_mwh`, the published complete
  day-ahead Hudson Valley LBMP for the target delivery hour.
- **Approved candidate predictor:** `load_forecast_mw`, selected from an
  archived forecast vintage available by the defined prediction cutoff.
- **Audit-only forecast fields:** forecast availability, cutoff, lead time, and
  source-provenance columns. They document the vintage selection but must not
  become model inputs.
- **Excluded provisional fields:** same-hour `actual_load_mw`, observed weather
  measurements and quality fields, plus the contemporaneous LMP component
  columns. Their availability or target relationship makes them unsuitable as
  operational day-ahead predictors at this stage.

In [141]:
target_column = "day_ahead_price_usd_mwh"

candidate_feature_columns = ["load_forecast_mw"]

forecast_audit_columns = [
    "forecast_available_at",
    "prediction_cutoff",
    "hours_before_cutoff",
    "source_archive",
    "source_file",
    "availability_basis",
    "availability_is_proxy",
    "forecast_horizon_hours",
]

print("Target:", target_column)
#print("Candidate features:", candidate_feature_columns)
#print("Audit columns:", forecast_audit_columns)

assert target_column in nyiso_electricity_with_load_forecasts.columns
assert set(candidate_feature_columns).issubset(
    nyiso_electricity_with_load_forecasts.columns
)

excluded_provisional_columns = [
    "actual_load_mw",
    "temperature_c",
    "dew_point_c",
    "relative_humidity_pct",
    "wind_speed_mps",
    "loss_component_usd_mwh",
    "congestion_component_usd_mwh",
    "energy_component_usd_mwh",
]

assert set(excluded_provisional_columns).isdisjoint(
    candidate_feature_columns
)

#print("Column classification passed.")

Target: day_ahead_price_usd_mwh


In [142]:
candidate_feature_columns =["load_forecast_mw",]
print("Candidate predictors:", candidate_feature_columns)

Candidate predictors: ['load_forecast_mw']


In [143]:
forecast_audit_columns=[
     "forecast_available_at",
    "prediction_cutoff",
    "hours_before_cutoff",
    "source_archive",
    "source_file",
    "availability_basis",
    "availability_is_proxy",
    "forecast_horizon_hours",
]
print("Forcast Audit columns", forecast_audit_columns)

Forcast Audit columns ['forecast_available_at', 'prediction_cutoff', 'hours_before_cutoff', 'source_archive', 'source_file', 'availability_basis', 'availability_is_proxy', 'forecast_horizon_hours']


In [144]:
identifier_columns = [
    "timestamp_local",
    "timestamp_utc",
    "target_timestamp",
    "location_id",
    "location",
    "source_time_zone",
    "load_location",
    "load_location_id",
    "weather_station",
    "market",
    "pricing_location",
    "load_data_type",
]

print("Identifier columns:", identifier_columns)

Identifier columns: ['timestamp_local', 'timestamp_utc', 'target_timestamp', 'location_id', 'location', 'source_time_zone', 'load_location', 'load_location_id', 'weather_station', 'market', 'pricing_location', 'load_data_type']


In [145]:
excluded_operational_columns=[
    "loss_component_usd_mwh"
    ,"congestion_component_usd_mwh"
    ,"energy_component_usd_mwh"
    ,"actual_load_mw"
    ,"observed_at"
    ,"REPORT_TYPE"    
    ,"temperature_c"
    ,"dew_point_c"
    ,"relative_humidity_pct"
    ,"wind_speed_mps"
    ,"weather_quality_flagged"
    ,"weather_value_rejected"
    ,"weather_missing"
    ,"weather_imputed",
]
print("Excluded operational columns:", excluded_operational_columns)

column_groups = {
    "target": [target_column],
    "candidate_predictors": candidate_feature_columns,
    "forecast_audit": forecast_audit_columns,
    "identifiers": identifier_columns,
    "excluded_operational": excluded_operational_columns,
}

dataset_columns = set(nyiso_electricity_with_load_forecasts.columns)


for group_name, columns in column_groups.items():
    missing_columns= set(columns) - dataset_columns

    assert not missing_columns,(
    f"{group_name} contains missing columns: {sorted(missing_columns)}"
)

    assert len(columns) == len(set(columns)),(
    f"{group_name} contains duplicate column names"
)

print("Column-group existence and duplicate checks passed.")

Excluded operational columns: ['loss_component_usd_mwh', 'congestion_component_usd_mwh', 'energy_component_usd_mwh', 'actual_load_mw', 'observed_at', 'REPORT_TYPE', 'temperature_c', 'dew_point_c', 'relative_humidity_pct', 'wind_speed_mps', 'weather_quality_flagged', 'weather_value_rejected', 'weather_missing', 'weather_imputed']
Column-group existence and duplicate checks passed.


In [146]:
from itertools import combinations

for left_group, right_group in combinations(column_groups, 2):
    overlapping_columns = (
        set(column_groups[left_group])
        & set(column_groups[right_group])
    )

    assert not overlapping_columns, (
        f"{left_group} and {right_group} overlap: "
        f"{sorted(overlapping_columns)}"
    )

prohibited_operational_predictors = (
    set(excluded_operational_columns)
    | set(forecast_audit_columns)
    | {target_column}
)

selected_prohibited_columns = (
    set(candidate_feature_columns)
    & prohibited_operational_predictors
)

assert not selected_prohibited_columns, (
    "Prohibited operational predictors were selected: "
    f"{sorted(selected_prohibited_columns)}"
)

print("Column groups do not overlap.")
print("No prohibited operational predictors are selected.")

Column groups do not overlap.
No prohibited operational predictors are selected.


In [147]:
proxy_row_count = int(
    nyiso_electricity_with_load_forecasts["availability_is_proxy"].sum()
)

assert proxy_row_count == len(nyiso_electricity_with_load_forecasts)

assert (
    nyiso_electricity_with_load_forecasts["availability_basis"]
    == "zip_entry_last_modified"
).all()

print(
    "WARNING: load_forecast_mw uses a ZIP-entry last-modified-time "
    "availability proxy for all selected January 2025 hours."
)
print("Rows using the availability proxy:", proxy_row_count)

Rows using the availability proxy: 744


In [148]:
expected_january_target_hours = 31 * 24

assert len(nyiso_electricity_with_load_forecasts) == expected_january_target_hours

assert (
    nyiso_electricity_with_load_forecasts["timestamp_utc"].nunique()
    == expected_january_target_hours
)

assert nyiso_electricity_with_load_forecasts[target_column].notna().all()

print(
    "Selected dataset contains",
    expected_january_target_hours,
    "unique January 2025 target hours.",
)

Selected dataset contains 744 unique January 2025 target hours.


### Derive leakage-safe calendar features

Calendar features are determined by the target delivery hour and therefore are
known before the day-ahead prediction cutoff. They do not use actual load,
observed weather, or future electricity-price information.

In [149]:
nyiso_electricity_with_load_forecasts["hour_of_day"] = (
    nyiso_electricity_with_load_forecasts["timestamp_local"].dt.hour
)

nyiso_electricity_with_load_forecasts["day_of_week"] = (
    nyiso_electricity_with_load_forecasts["timestamp_local"].dt.dayofweek
)

nyiso_electricity_with_load_forecasts["is_weekend"] = (
    nyiso_electricity_with_load_forecasts["day_of_week"] >= 5
)

candidate_feature_columns = [
    "load_forecast_mw",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
]

assert nyiso_electricity_with_load_forecasts["hour_of_day"].between(0, 23).all()
assert nyiso_electricity_with_load_forecasts["day_of_week"].between(0, 6).all()
assert nyiso_electricity_with_load_forecasts["is_weekend"].notna().all()

assert pd.api.types.is_integer_dtype(
    nyiso_electricity_with_load_forecasts["hour_of_day"]
)
assert pd.api.types.is_integer_dtype(
    nyiso_electricity_with_load_forecasts["day_of_week"]
)
assert pd.api.types.is_bool_dtype(
    nyiso_electricity_with_load_forecasts["is_weekend"]
)

assert candidate_feature_columns == [
    "load_forecast_mw",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
]

print("Calendar-feature validation passed.")
print("Candidate features:", candidate_feature_columns)

Calendar-feature validation passed.
Candidate features: ['load_forecast_mw', 'hour_of_day', 'day_of_week', 'is_weekend']


In [150]:
assert (
    nyiso_electricity_with_load_forecasts["is_weekend"]
    == nyiso_electricity_with_load_forecasts["day_of_week"].isin([5, 6])
).all()

calendar_validation_summary = {
    "row_count": len(nyiso_electricity_with_load_forecasts),
    "hour_of_day_range": (
        nyiso_electricity_with_load_forecasts["hour_of_day"].min(),
        nyiso_electricity_with_load_forecasts["hour_of_day"].max(),
    ),
    "day_of_week_range": (
        nyiso_electricity_with_load_forecasts["day_of_week"].min(),
        nyiso_electricity_with_load_forecasts["day_of_week"].max(),
    ),
    "weekend_hours": int(
        nyiso_electricity_with_load_forecasts["is_weekend"].sum()
    ),
        "candidate_features": candidate_feature_columns,
}

calendar_validation_summary

{'row_count': 744,
 'hour_of_day_range': (np.int32(0), np.int32(23)),
 'day_of_week_range': (np.int32(0), np.int32(6)),
 'weekend_hours': 192,
 'candidate_features': ['load_forecast_mw',
  'hour_of_day',
  'day_of_week',
  'is_weekend']}

In [151]:
example_target_timestamp = pd.Timestamp(
    "2025-01-10 12:00",
    tz="America/New_York",
)

example_prediction_cutoff = pd.Timestamp(
    "2025-01-09 05:00",
    tz="America/New_York",
)

naive_lag_source_target_hour = (
    example_target_timestamp - pd.Timedelta(hours=1)
)

print("Target delivery hour:", example_target_timestamp)
print("Prediction cutoff:", example_prediction_cutoff)
print("Naive one-hour lag source:", naive_lag_source_target_hour)

assert naive_lag_source_target_hour > example_prediction_cutoff

print("Counterexample passed: the preceding row is after the cutoff.")

Target delivery hour: 2025-01-10 12:00:00-05:00
Prediction cutoff: 2025-01-09 05:00:00-05:00
Naive one-hour lag source: 2025-01-10 11:00:00-05:00
Counterexample passed: the preceding row is after the cutoff.


In [152]:
marker_cutoff_hours ={
    "NYISO":5,
    "PJM":11
}
provisional_latest_price_available_hour =0

print("Market cutoff hours: ", marker_cutoff_hours)
print("Provisional latest day-ahead price availability hour:",
provisional_latest_price_available_hour,)
    

Market cutoff hours:  {'NYISO': 5, 'PJM': 11}
Provisional latest day-ahead price availability hour: 0


In [153]:
nyiso_electricity_with_load_forecasts[
    "day_ahead_price_available_at"
] = (
    nyiso_electricity_with_load_forecasts["timestamp_local"].dt.normalize()
    + pd.Timedelta(
        hours=provisional_latest_price_available_hour
    )
)

nyiso_electricity_with_load_forecasts[
    [
        "timestamp_local",
        "day_ahead_price_available_at",
    ]
].head()

,timestamp_local,day_ahead_price_available_at
0,2025-01-01 00:00:00-05:00,2025-01-01 00:00:00-05:00
1,2025-01-01 01:00:00-05:00,2025-01-01 00:00:00-05:00
2,2025-01-01 02:00:00-05:00,2025-01-01 00:00:00-05:00
3,2025-01-01 03:00:00-05:00,2025-01-01 00:00:00-05:00
4,2025-01-01 04:00:00-05:00,2025-01-01 00:00:00-05:00


In [154]:
"""
    This cell validates the new day_ahead_price_available_at audit column, then displays one important example.
    The four assert checks mean:
    Every row has a price-availability time.
    The availability times include a timezone.
    Each availability time is midnight on the same date as its price’s delivery hour.
    An availability time is never after its own delivery hour.

    SELECT timestamp_local,
       day_ahead_price_available_at,
       prediction_cutoff
    FROM nyiso_electricity_with_load_forecasts
    WHERE timestamp_local = '2025-01-10 12:00:00-05:00';

    Because midnight on January 10 is after 5:00 a.m. on January 9, the January 10 price schedule 
    is not available when the January 10 forecast is made.
    Next, create a Boolean audit column named target_day_price_available_by_cutoff. It answers:
    “Is this row’s own target-day price available at this row’s forecast cutoff?”

    We expect the answer to be False for all 744 rows. That is good—it proves we are not accidentally 
    using the target-day price itself as a feature.

"""

assert (
    nyiso_electricity_with_load_forecasts[
        "day_ahead_price_available_at"
    ].notna().all()
)

assert (
    nyiso_electricity_with_load_forecasts[
        "day_ahead_price_available_at"
    ].dt.tz
    is not None
)

assert (
    nyiso_electricity_with_load_forecasts[
        "day_ahead_price_available_at"
    ].dt.normalize()
    == nyiso_electricity_with_load_forecasts[
        "timestamp_local"
    ].dt.normalize()
).all()

assert (
    nyiso_electricity_with_load_forecasts[
        "day_ahead_price_available_at"
    ]
    <= nyiso_electricity_with_load_forecasts["timestamp_local"]
).all()

nyiso_electricity_with_load_forecasts.loc[
    nyiso_electricity_with_load_forecasts["timestamp_local"]
    == example_target_timestamp,
    [
        "timestamp_local",
        "day_ahead_price_available_at",
        "prediction_cutoff",
    ],
]

,timestamp_local,day_ahead_price_available_at,prediction_cutoff
228,2025-01-10 12:00:00-05:00,2025-01-10 00:00:00-05:00,2025-01-09 05:00:00-05:00


In [155]:
nyiso_electricity_with_load_forecasts[
    "target_day_price_available_by_cutoff"
] = (
    nyiso_electricity_with_load_forecasts[
        "day_ahead_price_available_at"
    ]
    <= nyiso_electricity_with_load_forecasts["prediction_cutoff"]
)

assert not nyiso_electricity_with_load_forecasts[
    "target_day_price_available_by_cutoff"
].any()

print(
    "Target-day prices available by cutoff:",
    int(
        nyiso_electricity_with_load_forecasts[
            "target_day_price_available_by_cutoff"
        ].sum()
    ),
)

Target-day prices available by cutoff: 0


In [156]:
nyiso_electricity_with_load_forecasts[
    "previous_day_same_hour_source_timestamp"
]=(
    nyiso_electricity_with_load_forecasts['timestamp_local']-pd.DateOffset(days=1)
)

nyiso_electricity_with_load_forecasts.loc[
    nyiso_electricity_with_load_forecasts["timestamp_local"]
    ==example_target_timestamp,
    [
        "timestamp_local",
        "previous_day_same_hour_source_timestamp",
        "prediction_cutoff",
    ],
]


,timestamp_local,previous_day_same_hour_source_timestamp,prediction_cutoff
228,2025-01-10 12:00:00-05:00,2025-01-09 12:00:00-05:00,2025-01-09 05:00:00-05:00


In [157]:
# Self-join each target hour to the previous delivery day's same-hour price
# and its provisional availability timestamp. Keep these as audit fields
# until they pass the target forecast-cutoff check.

price_source_lookup = (
    nyiso_electricity_with_load_forecasts[
        [
            "timestamp_local",
            target_column,
            "day_ahead_price_available_at",
        ]
    ]
    .rename(
        columns={
            "timestamp_local": (
                "previous_day_same_hour_source_timestamp"
            ),
            target_column: "previous_day_same_hour_price_value",
            "day_ahead_price_available_at": (
                "previous_day_same_hour_price_available_at"
            ),
        }
    )
)

nyiso_with_price_sources = (
    nyiso_electricity_with_load_forecasts
    .merge(
        price_source_lookup,
        how="left",
        on="previous_day_same_hour_source_timestamp",
        validate="one_to_one",
    )
)

assert len(nyiso_with_price_sources) == len(
    nyiso_electricity_with_load_forecasts
)

nyiso_with_price_sources.loc[
    nyiso_with_price_sources["timestamp_local"]
    == example_target_timestamp,
    [
        "timestamp_local",
        "prediction_cutoff",
        "previous_day_same_hour_source_timestamp",
        "previous_day_same_hour_price_value",
        "previous_day_same_hour_price_available_at",
    ],
]

,timestamp_local,prediction_cutoff,previous_day_same_hour_source_timestamp,previous_day_same_hour_price_value,previous_day_same_hour_price_available_at
228,2025-01-10 12:00:00-05:00,2025-01-09 05:00:00-05:00,2025-01-09 12:00:00-05:00,131.68,2025-01-09 00:00:00-05:00


In [158]:
# Mark prior-day same-hour source prices that were available by each target cutoff.

nyiso_with_price_sources[
    "previous_day_same_hour_price_is_available"
] = (
    nyiso_with_price_sources[
        "previous_day_same_hour_price_available_at"
    ].notna()
    & (
        nyiso_with_price_sources[
            "previous_day_same_hour_price_available_at"
        ]
        <= nyiso_with_price_sources["prediction_cutoff"]
    )
)

has_previous_day_source = nyiso_with_price_sources[
    "previous_day_same_hour_price_value"
].notna()

assert nyiso_with_price_sources.loc[
    has_previous_day_source,
    "previous_day_same_hour_price_is_available",
].all()

print(
    "Rows with a prior-day same-hour source:",
    int(has_previous_day_source.sum()),
)
print(
    "Availability-safe prior-day same-hour sources:",
    int(
        nyiso_with_price_sources[
            "previous_day_same_hour_price_is_available"
        ].sum()
    ),
)

Rows with a prior-day same-hour source: 720
Availability-safe prior-day same-hour sources: 720


In [159]:
# prove that each source price is available by the target cutoff before we allow it to become a predictor.
from electricity_forecasting.feature_engineering import (
    is_available_by_cutoff,
)

nyiso_with_price_sources[
    "previous_day_same_hour_price_is_available"
] = is_available_by_cutoff(
    nyiso_with_price_sources[
        "previous_day_same_hour_price_available_at"
    ],
    nyiso_with_price_sources["prediction_cutoff"],
)

has_previous_day_source = nyiso_with_price_sources[
    "previous_day_same_hour_price_value"
].notna()

assert nyiso_with_price_sources.loc[
    has_previous_day_source,
    "previous_day_same_hour_price_is_available",
].all()

print(
    "Rows with a prior-day same-hour source:",
    int(has_previous_day_source.sum()),
)
print(
    "Availability-safe prior-day same-hour sources:",
    int(
        nyiso_with_price_sources[
            "previous_day_same_hour_price_is_available"
        ].sum()
    ),
)

Rows with a prior-day same-hour source: 720
Availability-safe prior-day same-hour sources: 720


In [160]:
from electricity_forecasting.feature_engineering import (
    add_cutoff_safe_feature,
)

nyiso_with_price_sources = add_cutoff_safe_feature(
    nyiso_with_price_sources,
    source_value_column="previous_day_same_hour_price_value",
    source_available_at_column=(
        "previous_day_same_hour_price_available_at"
    ),
    prediction_cutoff_column="prediction_cutoff",
    feature_column="day_ahead_price_lag_1d",
)

assert (
    nyiso_with_price_sources["day_ahead_price_lag_1d"].notna()
    == nyiso_with_price_sources[
        "previous_day_same_hour_price_is_available"
    ]
).all()

print(
    "Nonmissing cutoff-safe prior-day price lags:",
    int(
        nyiso_with_price_sources[
            "day_ahead_price_lag_1d"
        ].notna().sum()
    ),
)

Nonmissing cutoff-safe prior-day price lags: 720


In [161]:
# Average the last 24 cutoff-safe prior-day price values.
# The target hour's own price is never an input to this calculation.
from electricity_forecasting.feature_engineering import (
    add_rolling_mean_from_safe_feature,
)

nyiso_with_price_sources = (
    add_rolling_mean_from_safe_feature(
        nyiso_with_price_sources,
        safe_feature_column="day_ahead_price_lag_1d",
        window=24,
        feature_column=(
            "day_ahead_price_lag_1d_rolling_mean_24h"
        ),
    )
)

print(
    "Nonmissing 24-hour rolling means:",
    int(
        nyiso_with_price_sources[
            "day_ahead_price_lag_1d_rolling_mean_24h"
        ].notna().sum()
    ),
)

Nonmissing 24-hour rolling means: 697


In [162]:
#Makes one cell responsible for approving both related price features, in the intended order. 
# The for loop is like a C# foreach over two strings, with a duplicate guard before .Add(...).

for feature_column in [
    "day_ahead_price_lag_1d",
    "day_ahead_price_lag_1d_rolling_mean_24h",
]:
    if feature_column not in candidate_feature_columns:
        candidate_feature_columns.append(feature_column)

assert set(
    [
        "day_ahead_price_lag_1d",
        "day_ahead_price_lag_1d_rolling_mean_24h",
    ]
).issubset(nyiso_with_price_sources.columns)

assert len(candidate_feature_columns) == len(
    set(candidate_feature_columns)
)
assert set(candidate_feature_columns).isdisjoint(
    prohibited_operational_predictors
)

print("Candidate predictors:", candidate_feature_columns)

Candidate predictors: ['load_forecast_mw', 'hour_of_day', 'day_of_week', 'is_weekend', 'day_ahead_price_lag_1d', 'day_ahead_price_lag_1d_rolling_mean_24h']


In [163]:
rolling_feature = (
    "day_ahead_price_lag_1d_rolling_mean_24h"
)

first_full_rolling_position = 47

safe_values_used = nyiso_with_price_sources[
    "day_ahead_price_lag_1d"
].iloc[24:48]

observed_rolling_mean = nyiso_with_price_sources[
    rolling_feature
].iloc[first_full_rolling_position]

assert len(safe_values_used) == 24
assert safe_values_used.notna().all()
assert abs(
    observed_rolling_mean - safe_values_used.mean()
) < 1e-10

print(
    "First full rolling mean uses 24 cutoff-safe lag values."
)

First full rolling mean uses 24 cutoff-safe lag values.


In [164]:
first_delivery_day = (
    nyiso_with_price_sources["timestamp_local"].dt.normalize()
    == pd.Timestamp("2025-01-01", tz="America/New_York")
)

assert first_delivery_day.sum() == 24
assert nyiso_with_price_sources.loc[
    first_delivery_day,
    "day_ahead_price_lag_1d",
].isna().all()
assert nyiso_with_price_sources.loc[
    ~first_delivery_day,
    "day_ahead_price_lag_1d",
].notna().all()

print(
    "First delivery-day rows without a prior-day source:",
    int(first_delivery_day.sum()),
)

First delivery-day rows without a prior-day source: 24


In [165]:
# Next action: add one validation cell to prove every target row asks for exactly the prior calendar day’s same-hour source—not merely any earlier row.
# This is like asserting a SQL join key is always target_timestamp - 1 calendar day. DateOffset(days=1) means the same local clock hour on the preceding
# date, which is the intended “previous-day same-hour” baseline

expected_previous_day_source = (
    nyiso_with_price_sources["timestamp_local"]
    - pd.DateOffset(days=1)
)

assert nyiso_with_price_sources[
    "previous_day_same_hour_source_timestamp"
].eq(expected_previous_day_source).all()

print(
    "Previous-day same-hour source timestamps verified:",
    len(nyiso_with_price_sources),
)

Previous-day same-hour source timestamps verified: 744


In [166]:
price_feature_audit_columns = [
    "day_ahead_price_available_at",
    "target_day_price_available_by_cutoff",
    "previous_day_same_hour_source_timestamp",
    "previous_day_same_hour_price_value",
    "previous_day_same_hour_price_available_at",
    "previous_day_same_hour_price_is_available",
]

for audit_column in price_feature_audit_columns:
    if audit_column not in forecast_audit_columns:
        forecast_audit_columns.append(audit_column)

column_groups = {
    "target": [target_column],
    "candidate_predictors": candidate_feature_columns,
    "forecast_audit": forecast_audit_columns,
    "identifiers": identifier_columns,
    "excluded_operational": excluded_operational_columns,
}

dataset_columns = set(nyiso_with_price_sources.columns)

for group_name, columns in column_groups.items():
    missing_columns = set(columns) - dataset_columns

    assert not missing_columns, (
        f"{group_name} contains missing columns: "
        f"{sorted(missing_columns)}"
    )
    assert len(columns) == len(set(columns)), (
        f"{group_name} contains duplicate column names"
    )

prohibited_operational_predictors = (
    set(excluded_operational_columns)
    | set(forecast_audit_columns)
    | {target_column}
)

assert set(candidate_feature_columns).isdisjoint(
    prohibited_operational_predictors
)

print("Price-source audit columns:", price_feature_audit_columns)
print("Final column-group checks passed.")

Price-source audit columns: ['day_ahead_price_available_at', 'target_day_price_available_by_cutoff', 'previous_day_same_hour_source_timestamp', 'previous_day_same_hour_price_value', 'previous_day_same_hour_price_available_at', 'previous_day_same_hour_price_is_available']
Final column-group checks passed.


In [167]:
from itertools import combinations

for left_group, right_group in combinations(column_groups, 2):
    overlapping_columns = (
        set(column_groups[left_group])
        & set(column_groups[right_group])
    )

    assert not overlapping_columns, (
        f"{left_group} and {right_group} overlap: "
        f"{sorted(overlapping_columns)}"
    )

print("Final column groups do not overlap.")

Final column groups do not overlap.


In [168]:
assert set(pjm_candidate_feature_columns) == set(
    common_candidate_feature_columns
)

assert set(candidate_feature_columns) == set(
    nyiso_augmented_candidate_feature_columns
)

assert set(nyiso_augmented_candidate_feature_columns) == (
    set(pjm_candidate_feature_columns)
    | {"load_forecast_mw"}
)

print(
    "Common PJM/NYISO feature-set check passed."
)
print(
    "NYISO augmented feature set adds only: load_forecast_mw"
)

Common PJM/NYISO feature-set check passed.
NYISO augmented feature set adds only: load_forecast_mw
